# Lesson 08: Replicating a Research Paper - Vision Transformer (ViT)

This notebook replicates the Vision Transformer (ViT) from the paper:
**"An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale"**
by Dosovitskiy et al. (Google Research, 2020)

## What We'll Build:
- Vision Transformer (ViT) from scratch in PyTorch
- Patch embedding layer
- Multi-head self-attention
- Transformer encoder blocks
- Complete ViT architecture

## Key Innovation of ViT:
**Transformers** (originally for text) applied directly to **images**!

### How it works:
1. **Split image into patches** (e.g., 16x16 pixels)
2. **Flatten patches** into vectors (like words in a sentence)
3. **Add positional information** (where each patch came from)
4. **Pass through Transformer** (self-attention learns relationships)
5. **Classify image** (from the learned representation)

### Why it's revolutionary:
- No convolutions needed!
- Attention learns global relationships (CNNs are local)
- Scales amazingly with data
- State-of-the-art results on ImageNet

## Paper Reading Tips:
1. **Abstract**: What problem? What solution?
2. **Introduction**: Why is this important?
3. **Methods**: How does it work?
4. **Figures**: Visual understanding
5. **Results**: Does it work?
6. **Skip math on first read** - implement it instead!

## Key Equations from Paper:
1. **Patch Embedding**: Image → Patches → Embeddings
2. **Multi-Head Attention**: Learning relationships between patches
3. **MLP**: Non-linear transformations
4. **Transformer Encoder**: Stacking attention + MLP blocks

## Setup and Imports

In [ ]:
# Check versions - this notebook requires PyTorch 1.12+ and torchvision 0.13+
try:
    import torch
    import torchvision
    
    # Assert version requirements
    torch_major, torch_minor = int(torch.__version__.split(".")[0]), int(torch.__version__.split(".")[1])
    torchvision_minor = int(torchvision.__version__.split(".")[1])
    
    assert torch_major >= 2 or (torch_major == 1 and torch_minor >= 12), "torch version should be 1.12+"
    assert torchvision_minor >= 13, "torchvision version should be 0.13+"
    
    print(f"✓ torch version: {torch.__version__}")
    print(f"✓ torchvision version: {torchvision.__version__}")
except:
    print("[WARNING] torch/torchvision versions may be incompatible")

In [ ]:
# Continue with regular imports
import matplotlib.pyplot as plt
import torch
import torchvision

from torch import nn
from torchvision import transforms

# Try to get torchinfo (for model summaries)
try:
    from torchinfo import summary
except:
    print("[INFO] Couldn't find torchinfo... installing it.")
    !pip install -q torchinfo
    from torchinfo import summary

# Import our modular code
try:
    from going_modular import data_setup, engine
    from going_modular.utils import download_data
except:
    print("[INFO] Couldn't find going_modular... downloading from GitHub.")
    !git clone https://github.com/mrdbourke/pytorch-deep-learning
    !mv pytorch-deep-learning/going_modular .
    !rm -rf pytorch-deep-learning
    from going_modular import data_setup, engine
    from going_modular.utils import download_data

In [ ]:
# Setup device-agnostic code
# ViT benefits greatly from GPU acceleration!
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Calculations will be done on: {device}")

In [ ]:
# Check GPU if available
!nvidia-smi

In [ ]:
def set_seeds(seed: int=42):
    """
    Sets random seeds for torch operations.
    
    Critical for reproducibility when replicating papers!

    Args:
        seed (int, optional): Random seed to set. Defaults to 42.
    """
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

## 1. Get Data

In [ ]:
# Download pizza, steak, sushi images
# Note: ViT paper uses ImageNet (1.2M images, 1000 classes)
# We'll use a smaller dataset for demonstration
image_path = download_data(
    source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
    destination="pizza_steak_sushi"
)
print(image_path)

In [ ]:
# Setup train and test directories
train_dir = image_path / "train"
test_dir = image_path / "test"

print(train_dir)
print(test_dir)

## 2. Create Datasets and DataLoaders

In [ ]:
# Image size from the ViT paper
# Original ViT uses 224x224 (can also use 384x384 for ViT-L/16)
IMG_SIZE = 224

# Create transform pipeline
# Note: ViT paper uses more sophisticated augmentation
# We'll keep it simple for demonstration
manual_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),  # [0, 255] → [0, 1]
])

print(f"Manually created transforms: {manual_transforms}")

In [ ]:
# Set batch size
# Note: ViT paper uses batch size of 4096 on TPUs!
# We use 32 because we're starting small
BATCH_SIZE = 32

# Create DataLoaders
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=manual_transforms,
    batch_size=BATCH_SIZE
)

print(f"Training batches: {len(train_dataloader)}")
print(f"Testing batches: {len(test_dataloader)}")
print(f"Classes: {class_names}")

In [ ]:
# Get a sample batch to work with
image_batch, label_batch = next(iter(train_dataloader))

# Get a single image and label
image, label = image_batch[0], label_batch[0]

print(f"Single image shape: {image.shape}")  # [C, H, W]
print(f"Single label: {label} ({class_names[label]})")

In [ ]:
# Visualize a sample image
plt.imshow(image.permute(1, 2, 0))  # [C, H, W] → [H, W, C] for matplotlib
plt.title(class_names[label])
plt.axis(False)
plt.show()

## 3. Understanding Vision Transformer Architecture

### Key Components:

1. **Patch Embedding**
   - Split image into patches (e.g., 16x16 pixels)
   - Flatten each patch
   - Project to embedding dimension

2. **Position Embedding**
   - Add learnable position info to each patch
   - Tells model where patches came from

3. **Class Token**
   - Special learnable token prepended to sequence
   - Used for final classification

4. **Transformer Encoder**
   - Multi-head self-attention (learn patch relationships)
   - MLP (non-linear transformation)
   - Layer normalization
   - Residual connections

5. **Classification Head**
   - MLP on class token output
   - Predicts class probabilities

## 4. Equation 1: Patch Embedding

**Goal**: Transform 2D image → 1D sequence of patch embeddings

**Math from paper**:
```
z₀ = [x_class; x_p^1 E; x_p^2 E; ...; x_p^N E] + E_pos
```

Where:
- `x_class`: Class token (learned)
- `x_p`: Flattened patches
- `E`: Embedding matrix (linear projection)
- `E_pos`: Position embeddings (learned)
- `N`: Number of patches

### 4.1 Calculating Patch Embedding Dimensions

In [ ]:
# Create example values from ViT paper
height = 224  # Image height
width = 224   # Image width
color_channels = 3  # RGB
patch_size = 16  # From paper: ViT-Base/16 (16x16 patches)

# Calculate number of patches
# Formula: N = (H × W) / P²
# where H=height, W=width, P=patch_size
number_of_patches = int((height * width) / patch_size**2)
print(f"Number of patches: {number_of_patches}")
print(f"Calculation: ({height} × {width}) / {patch_size}² = {number_of_patches}")

In [ ]:
# Input and output shapes for patch embedding

# Input: 2D image
embedding_layer_input_shape = (height, width, color_channels)

# Output: 1D sequence of flattened patches
# Each patch: patch_size² × color_channels = 16² × 3 = 768 values
embedding_layer_output_shape = (number_of_patches, patch_size**2 * color_channels)

print(f"Input shape (single 2D image): {embedding_layer_input_shape}")
print(f"Output shape (1D sequence of patches): {embedding_layer_output_shape}")
print(f"\nEach patch has {patch_size**2 * color_channels} values (16×16×3)")

### 4.2 Visualizing Image Patches

Let's see what these patches actually look like!

In [ ]:
# Show the full image
print(f"Image shape: {image.shape}")  # [3, 224, 224]

plt.imshow(image.permute(1, 2, 0))
plt.title(class_names[label])
plt.axis(False)
plt.show()

In [ ]:
# Get the top row of the image (first 16 pixels in height)
image_permuted = image.permute(1, 2, 0)  # [C, H, W] → [H, W, C]

# Visualize the first row of patches
patch_size = 16
plt.figure(figsize=(patch_size, patch_size))
plt.imshow(image_permuted[:patch_size, :, :])  # First 16 rows
plt.title(f"Top {patch_size} rows of image")
plt.axis(False)
plt.show()

print(f"This represents the top row of {int(224/16)} patches")

In [ ]:
# Visualize individual patches from the top row
img_size = 224
patch_size = 16
num_patches = img_size / patch_size

assert img_size % patch_size == 0, "Image size must be divisible by patch size"

print(f"Number of patches per row: {num_patches}")
print(f"Patch size: {patch_size} × {patch_size} pixels")

# Create subplots for each patch in top row
fig, axs = plt.subplots(nrows=1, ncols=int(num_patches), figsize=(num_patches*2, 2))

# Loop through each patch in the top row
for i, ax in enumerate(axs):
    # Get the patch
    start_col = i * patch_size
    end_col = start_col + patch_size
    
    # Extract patch: [rows 0-16, columns start:end, all colors]
    patch = image_permuted[:patch_size, start_col:end_col, :]
    
    # Plot
    ax.imshow(patch)
    ax.set_title(f"Patch {i+1}")
    ax.axis("off")

plt.suptitle("Top row of image split into patches")
plt.show()

print(f"\nEach patch is {patch_size}×{patch_size}×{color_channels} = {patch_size**2 * color_channels} values")